### 11_ElasticNet_Feature selection_Pearson_MLU

## 1 Importer les outils et charger le fichier


In [ ]:
# import packages 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Pretraitement, preprocessing modelisation
from sklearn import model_selection, preprocessing

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import ElasticNet, LinearRegression

from sklearn.model_selection import cross_val_predict, cross_val_score, cross_validate, train_test_split


#Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


#others
from pathlib import Path
import joblib
from sklearn.pipeline import Pipeline






In [ ]:
#Chargement de fichier commun

# fichier fusionné par le groupe (07_Merge)
df = pd.read_csv('../data/final_metropole/Merge_final_2.csv', dtype={'code_dept': str})

display(df.info())

df.columns.tolist()

cible = 'TousAge_allergie_urgences'
df[cible].describe()

Supression des colonnes

In [ ]:
# suppression des colonnes autre cible que taux_urgence_allergie_tous_age,  les categories Ntop et Npop

colonnes_supprimer = ["65plus_allergie_urgences", "0_14_allergie_urgences" , "15_64_allergie_urgences",
                         "0_14_asthme_urgences","15_64_asthme_urgences",  "TousAge_asthme_urgences", "65plus_asthme_urgences",
                         "0an_bronchiolite_urgences","TousAge_allergie_hospitalisation", "65plus_allergie_hospitalisation",
                         "0_14_allergie_hospitalisation", "15_64_allergie_hospitalisation", "0_14_asthme_hospitalisation", 
                         "15_64_asthme_hospitalisation", "TousAge_asthme_hospitalisation", "65plus_asthme_hospitalisation", 
                         "0an_bronchiolite_hospitalisation", "nom_dep","Ntop_Maladie_Respi_Chronique_tsage_9", 'Npop_Maladie_Respi_Chronique_tsage_9', 
                         'Ntop_Maladie_Respi_Chronique_0-14_9', 'Ntop_Maladie_Respi_Chronique_15-64_9',
                         'Ntop_Maladie_Respi_Chronique_65+_9', 'Npop_Maladie_Respi_Chronique_0-14_9', 'Npop_Maladie_Respi_Chronique_15-64_9',
                         'Npop_Maladie_Respi_Chronique_65+_9', "prev_calc_Maladie_Respi_Chronique_0-14_9" , 
                         "prev_calc_Maladie_Respi_Chronique_15-64_9", "prev_calc_Maladie_Respi_Chronique_65+_9"]

df_work = df.drop(columns = colonnes_supprimer).copy()

df_work.info()

In [ ]:
# inventaire des colonnes


# Colonnes numériques
cols_num = df_work.select_dtypes(include="number").columns.tolist()

# Colonnes catégorielles ou dates stockées en texte
cols_cat = df_work.select_dtypes(include=["object", "category"]).columns.tolist()

# Colonnes datetime
cols_date = df_work.select_dtypes(include="datetime").columns.tolist()

print("Numériques :", len(cols_num), cols_num)
print("Catégorielles :", len(cols_cat), cols_cat)
print("Dates :", len(cols_date), cols_date)

## 2 Encodage cyclic par sin /cos
On conserve le code qui extrait `annee` et `semaine` du libellé `2021-S01`.
Le tri se fait sur département, année et numéro de semaine. Aucune conversion en date n’est nécessaire.
Les blocs de quatre semaines sont calculés comme dans son notebook.


In [ ]:

# extraire l'année et semaine,
df_work["annee"] = df_work["date_semaine"].str[:4].astype(int)
df_work["semaine"] = df_work["date_semaine"].str[-2:].astype(int)

# Représenter la saisonnalité annuelle par un cycle de 52 semaines

df_work["semaine_sin"] = np.sin(2*np.pi*df_work["semaine"]/52)
df_work["semaine_cos"] = np.cos(2*np.pi*df_work["semaine"]/52)

#trier par année, semaine et departement
df_work = df_work.sort_values(["code_dept", "annee", "semaine"]).reset_index(drop=True)
assert not df_work.duplicated(["code_dept", "annee", "semaine"]).any()                      #verifier si il y a des doublons

## 3 Créer le taux à prévoir
Sur une ligne de semaine X, on garde les informations de semaine X et on place le taux de semaine X+1 dans une nouvelle colonne `taux_suivant`.
`shift(-1)` place la réponse future sur la ligne des informations courantes. Il est calculé séparément par département pour ne pas mélanger leurs séries.

`taux_connu` reprend le taux courant. **On suppose ici qu’il est déjà publié à la date de prévision.** Si ce n’est pas le cas, il faudra utiliser un taux plus ancien.


In [ ]:
#creation des nouvelles colonnes pour les dates cibles et taux suivante 

df_work["taux_suivant"] = df_work.groupby("code_dept")[cible].shift(-1)       #regroupe les observations par departement et recupere la valeur de la ligne suivante dans la meme departement
df_work["taux_connu"] = df_work[cible]

# On rapproche aussi le libellé et le calendrier de la semaine à prévoir.
df_work["semaine_a_prevoir"] = df_work.groupby("code_dept")["date_semaine"].shift(-1)
df_work["annee_cible"] = df_work.groupby("code_dept")["annee"].shift(-1)
df_work["semaine_cible"] = df_work.groupby("code_dept")["semaine"].shift(-1)
df_work["semaine_cible_sin"] = df_work.groupby("code_dept")["semaine_sin"].shift(-1)
df_work["semaine_cible_cos"] = df_work.groupby("code_dept")["semaine_cos"].shift(-1)

In [ ]:
# La dernière semaine n’a pas de réponse suivante : on retire cette ligne, sans remplacer la cible par zéro.
data = df_work.dropna(subset=["taux_suivant", "taux_connu"]).copy()

data["annee_cible"] = data["annee_cible"].astype(int)
data["semaine_cible"] = data["semaine_cible"].astype(int)
data["semaine_cible_sin"] = data["semaine_cible_sin"].astype(float)
data["semaine_cible_cos"] = data["semaine_cible_cos"].astype(float)

data[["code_dept", "date_semaine", "semaine_a_prevoir", "taux_connu", "taux_suivant"]].head()


## Inventaire des colonnes

In [ ]:
# inventaire des colonnes sur data


# Colonnes numériques
cols_num = data.select_dtypes(include="number").columns.tolist()

# Colonnes catégorielles ou dates stockées en texte
cols_cat = data.select_dtypes(include=["object", "category"]).columns.tolist()

# Colonnes datetime
cols_date = data.select_dtypes(include="datetime").columns.tolist()

print("Numériques :", len(cols_num), cols_num)
print("Catégorielles :", len(cols_cat), cols_cat)
print("Dates :", len(cols_date), cols_date)

In [ ]:
resume = pd.DataFrame({
    "Type": data.dtypes,
    "Valeurs uniques": data.nunique(),
    "NaN (%)": data.isna().mean() * 100
})

display(resume)

### OPTIMISATION

## Feature selection

Objectif : supprimer des variables et vérifier le gain de prédiction.
Point de départ : **59 variables numériques + 3 categorielle**

Apprentissage 2021–2022 ; sélection des réglages des filtres sur quatre plis chronologiques en 2022 ;
contrôle descriptif sur 2023 ; **2024 réservé**. 

## 4 Séparer les années : train / valid / test
On regarde l’année **du taux à prévoir**. Fin 2022 pour prévoir début 2023 appartient à la validation.
A et B utiliseront les mêmes lignes. Dans les deux cas, la cible est le "taux_suivant"

In [ ]:
#Separation des années
data = data.copy()

train = data[data["annee_cible"].isin([2021, 2022])].copy()
valid = data[data["annee_cible"] == 2023].copy()
test = data[data["annee_cible"] == 2024].copy()

print("Train :", len(train), "lignes")
print("Validation :", len(valid), "lignes")
print("Test réservé :", len(test), "lignes")






## Methode de filtrage
1. Correlation de Pearson
(detection des correlation linaires et variables redondantes)

on repart sur "train" pourexaminer les correlations des variables
On retire les années/semaines brutes et la cible , en conservant le dernier taux connu. 

Je filtre sur on coefficient de correlation >0.8 pour selectionner les variables correlés à retirer

In [ ]:
#Selection des variables
# Sélection des colonnes numériques
train = train.copy()
num_cols = train.select_dtypes(include="number").columns

# Colonnes à exclure de l'analyse
exclure = [
    "annee", "semaine", "periode",
    "annee_cible", "semaine_cible", 
    "TousAge_allergie_urgences", "temps_poussiere_chasse_neige", 
    "semaine_cible_sin", "semaine_cible_cos",
    "semaine_sin", "semaine_cos"
]

# Exclure aussi les colonnes issues du encoding precedent
exclure += [col for col in num_cols if col.startswith("periode_")]

# Conserver les variables explicatives pertinentes
variables = [col for col in num_cols if col not in exclure]

print("Variables explicatives retenues :", len(variables), variables)

In [ ]:

# Calcul de matrices de correlation de Pearson
corr_pearson = train[variables].corr(method='pearson')

#print("Corrélations de Pearson :")
#display(corr_pearson)#*

# Garder uniquement le triangle supérieur (éviter les doublons)
masque = np.triu(np.ones(corr_pearson.shape), k=1).astype(bool)

# Transformer la matrice en liste de paires
paires_corr = corr_pearson.where(masque).stack()

# Filtrer les corrélations fortes et trier
paires_fortes = paires_corr[paires_corr.abs() > 0.8]
paires_fortes = paires_fortes.reindex(
    paires_fortes.abs().sort_values(ascending=False).index
)

display(paires_fortes)

Correlations fortes entre les variables > 0.8

1. NO3 in PM1 / NH4 in PM1 : Les nitrates (NO₃⁻) et l'ammonium (NH₄⁺) sont des composants des particules fines. Ils sont souvent associés chimiquement sous forme de nitrate d'ammonium, ce qui explique leur forte corrélation.  NO3 in PM1 à garder

2. tp_sans_precipitation_visibilite_ou_nuages/tp_temps_inconnu : tp_sans_precipitation_visible semble etre plus interpretable, à garder

3. PM10 /PM 2.5: Les PM10 regroupent les particules de diamètre aérodynamique inférieur ou égal à 10 µm, et les PM2.5 celles inférieures ou égales à 2,5 µm. Les PM2.5 constituent donc une partie des PM10. -- je garde PM10

4. pct_urbain / pct_agriculteur: correlation coherent - les territoires fortement urbanisés présentent généralement une plus faible proportion d'agriculteurs. Je garde pct_urbain                                 

In [ ]:
#Visualisation de la matrice de corrélation avec Plotly
import plotly.graph_objects as go

fig = go.Figure(
    data=go.Heatmap(
        z=corr_pearson.values,
        x=corr_pearson.columns,
        y=corr_pearson.index,
        colorscale="RdBu",
        zmin=-1, zmax=1,
        colorbar=dict(title="r", len=0.7)
    )
)

fig.update_layout(
    title="Matrice de corrélation de Pearson",
    width=1000,
    height=850,
    margin=dict(l=250, r=50, t=70, b=220),
    font=dict(size=9)
)

fig.update_yaxes(
    tickmode="linear",
    dtick=1,
    automargin=True,
    autorange="reversed"
)

fig.update_xaxes(
    tickangle=-90,
    tickfont=dict(size=8)
)

fig.show()

In [ ]:
#Train-test split
# Colonnes à exclure
colonnes_exclure = [
    "taux_suivant",
    "NH4 in PM1",
    "tp_temps_inconnu",
    "PM2.5",
    "pct_agriculteurs"
]

X_train = train.drop(columns=colonnes_exclure)
X_valid = valid.drop(columns=colonnes_exclure)
X_test = test.drop(columns=colonnes_exclure)

y_train = train["taux_suivant"]
y_valid = valid["taux_suivant"]
y_test = test["taux_suivant"]

In [ ]:
X_train.columns.tolist()

## 5. Encoder des departements

Encodeur ajuster sur le train, puis appliqué sur la validation et test

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Initialiser l'encodeur
enc = OneHotEncoder(drop="first", handle_unknown="ignore",
                    sparse_output=False).set_output(transform="pandas")

# Encoder les départements
X_train_enc = enc.fit_transform(X_train[["code_dept"]])
X_valid_enc = enc.transform(X_valid[["code_dept"]])
X_test_enc = enc.transform(X_test[["code_dept"]])
# Ajouter les périodes déjà encodées et enlever code_dept
X_train = pd.concat([X_train.drop(columns = "code_dept"), X_train_enc], axis=1)
X_valid = pd.concat([X_valid.drop(columns = "code_dept"), X_valid_enc], axis=1)
X_test = pd.concat([X_test.drop(columns = "code_dept"), X_test_enc], axis=1)

In [ ]:
X_train.info()
X_valid.info()
X_test.info()

In [ ]:
import joblib

# Sauvegarder l'encodeur
joblib.dump(enc, "encodeur_dept.joblib")

# Recharger l'encodeur plus tard
#enc = joblib.load("encodeur_dept.joblib")

# Encoder de nouvelles données
#new_enc = enc.transform(new_data[["code_dept"]])

## 6. Standardisation

Le scaler apprend ses moyennes et écarts-types sur **2021–2022 uniquement**.
En validation, `transform` réutilise ces nombres sans les recalculer.


In [ ]:
# Toutes les variables numériques à standardiser

# Sélectionner les variables numériques
numeriques = X_train.select_dtypes(include="number").columns

# Colonnes à exclure explicitement
exclure = [
    "annee", "semaine", "periode",
    "annee_cible", "semaine_cible", "periode_cible",
    "TousAge_allergie_urgences", "tp_tempete_poussiere_chasse_neige"
]

# Ajouter les colonnes binaires (0/1) et celles commençant par "periode_"
exclure += [
    col for col in numeriques
    if X_train[col].dropna().isin([0, 1]).all()
    or col.startswith("periode_")
]

# Conserver les variables à standardiser
variables_scaler = [col for col in numeriques if col not in exclure]

print("Colonnes à standardiser :", variables_scaler)

print("Nombre de colonnes numériques :", len(numeriques))
print("Colonnes numériques :", numeriques.tolist())

print("Colonnes exclues :", exclure)

In [ ]:
#instanciation de Scaler
scaler = StandardScaler().set_output(transform="pandas")

# Entrainer sur X_train variables_scaler (variables selectionné à standardiser)
scaler.fit(X_train[variables_scaler])

# Copier les DataFrames pour conserver les autres variables
X_train_scaled = X_train.copy()
X_valid_scaled = X_valid.copy()
X_test_scaled = X_test.copy()

# Standardiser uniquement les colonnes sélectionnées
X_train_scaled[variables_scaler] = scaler.transform(X_train[variables_scaler])
X_valid_scaled[variables_scaler] = scaler.transform(X_valid[variables_scaler])
X_test_scaled[variables_scaler] = scaler.transform(X_test[variables_scaler])




In [ ]:
#sauvegarder le scaler

joblib.dump(scaler, "scaler.joblib")
joblib.dump(variables_scaler, "variables_scaler.joblib")



In [ ]:
X_train_scaled.info()
X_valid_scaled.info()
X_test_scaled.info()

In [ ]:
print(X_train_scaled.select_dtypes(exclude="number").dtypes)
print(X_valid_scaled.select_dtypes(exclude="number").dtypes)
print(X_test_scaled.select_dtypes(exclude="number").dtypes)


In [ ]:
#supprimer les colonnes date_semaine, semaine_a_prevoir, TousAge_allergie_urgences, "semaine_cos", "semaine_sin" (semaine_cos, semaine_sin sont supprimées pour eviter la multicolinéarité)
X_train_scaled = X_train_scaled.drop(columns=["date_semaine", "semaine_a_prevoir", "TousAge_allergie_urgences", "semaine_cos", "semaine_sin"]).copy()
X_valid_scaled = X_valid_scaled.drop(columns=["date_semaine", "semaine_a_prevoir", "TousAge_allergie_urgences", "semaine_cos", "semaine_sin"]).copy()
X_test_scaled = X_test_scaled.drop(columns=["date_semaine", "semaine_a_prevoir", "TousAge_allergie_urgences", "semaine_cos", "semaine_sin"]).copy()

## 7 Choisir les colonnes utilisées pour A et B

In [ ]:
# A : sans historique sanitaire "taux_connu"
X_train_A = X_train_scaled.drop(columns=["taux_connu"]).copy()
X_valid_A = X_valid_scaled.drop(columns=["taux_connu"]).copy()


# B : avec historique sanitaire
X_train_B = X_train_scaled.copy()
X_valid_B = X_valid_scaled.copy()


In [ ]:

print(X_train_A.info())
print(X_train_B.info())

8. Fonction pour metrics

In [ ]:
def afficher_scores(nom, reel, prediction):
    print(nom)
    print("MAE  :", round(mean_absolute_error(reel, prediction), 2))
    print("RMSE :", round(mean_squared_error(reel, prediction) ** 0.5, 2))
    print("R²   :", round(r2_score(reel, prediction), 3))

9. BASELINE DE PERSISTANCE 

Prévision de la semaine suivante = dernier taux connu. Aucun modèle n’est entraîné.
Exemple : taux courant 700 → prévision 700 pour la semaine suivante.

On compare ensuite cette prévision au vrai taux de semaine 11 et on calcule MAE, RMSE et R², comme pour les modèles. Cela permet de vérifier si les modeles font mieux que simplement reprendre la dernière valeur. 

In [ ]:
prediction_persistance =valid["taux_connu"]
afficher_scores("Persistance — validation 2023", y_valid, prediction_persistance)

MAE = 219.03 : les prédictions s'écartent en moyenne de 204 unités du taux réel.
RMSE = 306.11 : 
R² = 0,104 : le modèle explique environ 10 % de la variabilité observée sur la validation.

## 10. Baseline saisonnière
Pour prévoir une semaine dans un département, on prend la moyenne de ses mêmes semaines dans le train.
La validation 2023 utilise donc les réponses de 2021–2022 uniquement.


In [ ]:
# Calculer le profil saisonnier
profil_saisonnier = (
    train.groupby(["code_dept", "semaine_cible"])["taux_suivant"]
    .mean()
    .rename("prediction_saisonniere")
)

# Recréer les prédictions
prediction_saisonniere = valid[
    ["code_dept", "semaine_cible"]
].join(
    profil_saisonnier,
    on=["code_dept", "semaine_cible"]
)["prediction_saisonniere"]

# Comparer
print(pd.DataFrame({
    "Réel": y_valid,
    "Prédit": prediction_saisonniere
}).head(10))

# Calculer les scores
afficher_scores("Baseline saisonnière", y_valid, prediction_saisonniere)

MAE = 203,92 : les prédictions s'écartent en moyenne de 204 unités du taux réel.
RMSE = 280,45 : 
R² = 0,248 : le modèle explique environ 25 % de la variabilité observée sur la validation.

Attention! Il n'y a que 2 données historiques par departement/semaine

## 11 Entraînement les deux ElasticNet
Les deux modèles (A et B) ont les mêmes réglages. La seule différence est la présence du taux passé dans B.
Les réglages ci-dessous sont un point de départ, pas un résultat d’optimisation.




In [ ]:

#model_A: sans "taux_connu"
modele_A = ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
modele_A.fit(X_train_A, y_train)
prediction_A_baseline = modele_A.predict(X_valid_A)

afficher_scores("ElasticNet A — validation 2023", y_valid, prediction_A_baseline)


In [ ]:
#model_B: avec taux_connu
modele_B = ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
modele_B.fit(X_train_B, y_train)
prediction_B_baseline = modele_B.predict(X_valid_B)

afficher_scores("ElasticNet B — validation 2023", y_valid, prediction_B_baseline)


Calcul de coefficients

In [ ]:
coefficients_A = pd.Series(
    modele_A.coef_,
    index=X_train_A.columns
)

coefficients_B = pd.Series(
    modele_B.coef_,
    index=X_train_B.columns
)

# Trier par importance (valeur absolue)
coefficients_A = coefficients_A.reindex(
    coefficients_A.abs().sort_values(ascending=False)[:15].index
)

display(coefficients_A)


coefficients_B = coefficients_B.reindex(
    coefficients_B.abs().sort_values(ascending=False)[:15].index
)

display(coefficients_B)

In [ ]:
coefficients_A = coefficients_A.round(3)
coefficients_B = coefficients_B.round(3)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

sns.barplot(x = coefficients_A.values , y = coefficients_A.index, color="#087E8B", ax=axes[0], orient = "h")
axes[0].set_title("ElasticNet A — Coefficients") 

sns.barplot(x = coefficients_B.values , y = coefficients_B.index, color="#71899E", ax=axes[1], orient = "h")
axes[1].set_title("ElasticNet B — Coefficients")

for ax in axes:
    ax.set_ylabel("Variables")



fig.suptitle("Coefficients des modèles — Validation 2023")
plt.tight_layout()
plt.show()

In [ ]:
#Summary of results

#calculer les scores per modele
def calculer_scores(nom, reel, prediction):
    return {
        "Modèle": nom,
        "MAE": mean_absolute_error(reel, prediction),
        "RMSE": mean_squared_error(reel, prediction) ** 0.5,
        "R²": r2_score(reel, prediction)
    }

# construction du tableau   

scores = pd.DataFrame([
    calculer_scores("Persistance", y_valid, prediction_persistance),
    calculer_scores("Saisonnier", y_valid, prediction_saisonniere),
    calculer_scores("ElasticNet A", y_valid, prediction_A_baseline),
    calculer_scores("ElasticNet B", y_valid, prediction_B_baseline)
])

# Affichage arrondi, sans modifier les valeurs conservées
display(scores.round({"MAE": 2, "RMSE": 2, "R²": 3}))

#barplot
metrique = "MAE"  # ou "RMSE"
couleurs = ["#A8B5C2", "#71899E", "#087E8B", "#087E8B"]


fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)

for ax, metrique in zip(axes, ["MAE", "RMSE", "R²"]):
    barres = ax.barh(
        scores["Modèle"],
        scores[metrique],
        color=couleurs,
        height=0.6
    )

    ax.bar_label(
        barres,
        labels=[
            f"{valeur:.2f}".replace(".", ",")
            for valeur in scores[metrique]
        ],
        padding=5,
        fontsize=11
    )

    ax.set_title(f"Comparaison des {metrique}")
    ax.set_xlabel(f"{metrique} (unités du taux)")

    maximum = scores[metrique].max()
    ax.set_xlim(0, maximum * 1.20 if maximum > 0 else 1)

    ax.spines[["top", "right"]].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(axis="x", alpha=0.2)

# Premier modèle en haut ; l'axe est partagé entre les deux graphiques
axes[0].invert_yaxis()

fig.suptitle("Performances des modèles apres selection de features avec Pearson — Validation 2023")
plt.tight_layout()
plt.show()

In [ ]:
#comparaison reel vs prediction


# Choisir le département
dept = "13"

# Regrouper les prédictions
comparaison = valid[["code_dept", "semaine_a_prevoir"]].copy()

comparaison["Taux_precedent"] = valid["taux_connu"]
comparaison["Réel (taux_suivant)"] = y_valid
comparaison["Persistance"] = prediction_persistance
#comparaison["Moyenne historique"] = prediction_saisonniere
comparaison["ElasticNet A"] = prediction_A
comparaison["ElasticNet B"] = prediction_B

# Filtrer le département et trier les semaines
courbes = comparaison[comparaison["code_dept"] == dept].sort_values("semaine_a_prevoir")

# Graphique
plt.figure(figsize=(14, 5))

for col in ["Taux_precedent", "Réel (taux_suivant)", "Persistance", "ElasticNet A", "ElasticNet B"]:
    plt.plot(courbes["semaine_a_prevoir"], courbes[col],
             label=col, linewidth=2 if col == "Réel" else 1.5)

plt.title(f"Prévisions vs valeurs réelles — Département {dept}")
plt.xlabel("Semaine cible")
plt.ylabel("Taux pour 100 000 passages")
plt.xticks(rotation=45)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:

# Comparaison réel vs prédictions

# Choisir le département
dept = "13"

# Regrouper les prédictions
comparaison = valid[["code_dept", "semaine_a_prevoir"]].copy()

comparaison["Taux_precedent"] = valid["taux_connu"]
comparaison["Réel (taux_suivant)"] = y_valid
comparaison["Persistance"] = prediction_persistance
comparaison["ElasticNet A"] = prediction_A
comparaison["ElasticNet B"] = prediction_B

# Filtrer le département et trier les semaines
courbes = comparaison[
    comparaison["code_dept"] == dept
].sort_values("semaine_a_prevoir")

# Deux graphiques superposés
fig, axes = plt.subplots(
    2, 1, figsize=(15, 8),
    sharex=True, sharey=True
)

modeles = [
    ("ElasticNet A", "#087E8B"),
    ("ElasticNet B", "#8B5EAD")
]

for ax, (modele, couleur) in zip(axes, modeles):

    ax.plot(
        courbes["semaine_a_prevoir"],
        courbes["Réel (taux_suivant)"],
        label="Réel",
        color="#E76F51",
        linewidth=2.5
    )

    ax.plot(
        courbes["semaine_a_prevoir"],
        courbes["Persistance"],
        label="Persistance",
        color="#A8B5C2",
        linestyle="--",
        linewidth=1.5,
        alpha=0.8
    )

    ax.plot(
        courbes["semaine_a_prevoir"],
        courbes[modele],
        label=modele,
        color=couleur,
        linewidth=2
    )

    ax.set_title(modele)
    ax.set_ylabel("Taux pour 100 000 passages")
    ax.legend(loc="upper right", ncol=3)
    ax.grid(alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)

# Afficher une semaine sur quatre
semaines = courbes["semaine_a_prevoir"].iloc[::4]

axes[-1].set_xticks(semaines)
axes[-1].tick_params(axis="x", rotation=45)
axes[-1].set_xlabel("Semaine cible")

fig.suptitle(
    f"Prévisions vs valeurs réelles — Département {dept} — 2023",
    fontsize=15
)

plt.tight_layout()
plt.show()


In [ ]:
colonnes = [
    "semaine_a_prevoir",
    "Réel (taux_suivant)",
    "Persistance",
    "ElasticNet A",
    "ElasticNet B"
]

display(courbes[colonnes].round(2))

on observe:
1. un décalage d'une semaine dans la prediction
2. le modele lisse les pics - pas adapté dans ce forme aux changements rapides

# C. OPTIMISATION DES HYPERPARAMETRES

## 13. Optimisation des hyperparametres avec GridSearchCV


### 13.a. Preparation des données

In [ ]:
# Je repars sur data et la separation des données

data = data.copy()

# je fais le trie par anné, semaine, departement
data = data.sort_values(
    ["annee_cible", "semaine_cible", "code_dept"]
).reset_index(drop=True)


#separation train - valid - test
train = data[data["annee_cible"].isin([2021, 2022])].copy()
valid = data[data["annee_cible"] == 2023].copy()
test = data[data["annee_cible"] == 2024].copy()

print("Train :", len(train), "lignes")
print("Validation :", len(valid), "lignes")
print("Test réservé :", len(test), "lignes")



In [ ]:
#Train-test split (X, y) et preparation set A et set B

#colonnes à exclure: date_semaine, semaine_a_prevoir, TousAge_allergie_urgences, "semaine_cos", "semaine_sin" 
# (semaine_cos, semaine_sin sont supprimées pour eviter la multicolinéarité)
exclure_X = [
    "taux_suivant",
    "date_semaine",
    "semaine_a_prevoir",
    "TousAge_allergie_urgences",
   "semaine_cos", "semaine_sin"
]

#Separation X, y A et B
X_train_B = train.drop(columns=exclure_X)
X_valid_B = valid.drop(columns=exclure_X)
X_test_B = test.drop(columns=exclure_X)

X_train_A = X_train_B.drop(columns="taux_connu")
X_valid_A = X_valid_B.drop(columns="taux_connu")
X_test_A = X_test_B.drop(columns="taux_connu")

y_train = train["taux_suivant"]
y_valid = valid["taux_suivant"]
y_test = test["taux_suivant"]

In [ ]:

#definition de la fonction de preprocessing pour les colonnes numeriques et categorielle (standardisation et encodage)

def creer_preprocessing(X):

# Colonnes numériques
    numeriques = X.select_dtypes(include="number").columns

# Colonnes à ne pas standardiser
    exclure = [
        "annee", "semaine",
        "tp_tempete_poussiere_chasse_neige",
        "annee_cible", "semaine_cible"
    ]

# Ajouter les colonnes binaires (0/1) et celles commençant par "periode_"
    exclure += [
        col for col in numeriques
        if X_train[col].dropna().isin([0, 1]).all()
        or col.startswith("periode_")
    ]
# Conserver les variables à standardiser
    variables_scaler = [col for col in numeriques if col not in exclure]

# Variables conservées sans transformation
    variables_autres = [
        col for col in X_train_B.columns
        if col not in variables_scaler + ["code_dept"]
    ]

    return ColumnTransformer(
        transformers=[
            ("scaler", StandardScaler(), variables_scaler),
            ("encoder", OneHotEncoder(
                drop="first",
                handle_unknown="ignore"
            ), ["code_dept"])
        ],
        remainder="passthrough"
    )


print("Nombre de colonnes numériques :", len(numeriques))
print("Colonnes numériques :", numeriques.tolist())
print("Colonnes à standardiser :", variables_scaler)
print("Colonnes  exclues de standardisation :", exclure)



### 13.b. Fold Temporelle

In [ ]:
#Creation de fold temporelle pour crossvalidation

folds_temporels = []

for debut, fin in [(1, 13), (14, 26), (27, 39), (40, 53)]:  #4 periodes de validation

    idx_train = np.where(
        (train["annee_cible"] == 2021) |
        ((train["annee_cible"] == 2022) &
         (train["semaine_cible"] < debut))
    )[0]

    idx_valid = np.where(
        (train["annee_cible"] == 2022) &
        (train["semaine_cible"].between(debut, fin))
    )[0]

    folds_temporels.append((idx_train, idx_valid))

In [ ]:
#verification des folds
for i, (idx_train, idx_valid) in enumerate(folds_temporels, 1):

    print(f"\nFold {i}")

    print("Train :", len(idx_train), "observations")
    print("Valid :", len(idx_valid), "observations")

    print("Années train :",
          train.iloc[idx_train]["annee_cible"].unique())

    print("Semaines validation :",
          train.iloc[idx_valid]["semaine_cible"].min(),
          "à",
          train.iloc[idx_valid]["semaine_cible"].max())

### 13.c. GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV

parameters = {
    "modele__alpha": [0.001, 0.01, 0.1, 1, 10],
    "modele__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
}

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

def optimiser_elasticnet(X, y):

    pipeline = Pipeline([
        ("preprocessing", creer_preprocessing(X)),
        ("modele", ElasticNet(max_iter=10000))
    ])

    grid=GridSearchCV(
        estimator = pipeline,
        param_grid = parameters,
        cv = folds_temporels,
        scoring = scoring,
        n_jobs = -1,
        refit = "MAE",
    )
    grid.fit(X, y)  

    return grid

# Optimisation des deux modèles
grid_A = optimiser_elasticnet(X_train_A, y_train)
grid_B = optimiser_elasticnet(X_train_B, y_train)


#Resultats
print("Meilleur parametres modele A:", grid_A.best_params_)
print("Meilleur score modele A:", grid_A.best_score_)

print("Meilleurs paramètres B :", grid_B.best_params_)
print("Meilleure MAE B :", grid_B.best_score_)





In [ ]:
prediction_A_opt = grid_A.predict(X_valid_A)
prediction_B_opt = grid_B.predict(X_valid_B)

afficher_scores("ElasticNet A optimisé", y_valid, prediction_A_opt)
afficher_scores("ElasticNet B optimisé", y_valid, prediction_B_opt)

### 13.d. Comparision des resultats

In [ ]:
#tableau comparatif des scores de 6 modeles sur validation 2023
def calculer_scores(nom, reel, prediction):
    return {
        "Modèle": nom,
        "MAE": mean_absolute_error(reel, prediction),
        "RMSE": mean_squared_error(reel, prediction) ** 0.5,
        "R²": r2_score(reel, prediction)
    }

scores = pd.DataFrame([
    calculer_scores("Persistance", y_valid, prediction_persistance),
    calculer_scores("Saisonnier", y_valid, prediction_saisonniere),
    calculer_scores("ElasticNet A baseline", y_valid, prediction_A),
    calculer_scores("ElasticNet B baseline", y_valid, prediction_B),
    calculer_scores("ElasticNet A optimisé", y_valid, prediction_A_opt),
    calculer_scores("ElasticNet B optimisé", y_valid, prediction_B_opt)
])

display(scores.round(3))

In [ ]:
scores = scores.sort_values("MAE").reset_index(drop=True)

display(scores.round(3))

In [ ]:
#barplot
metriques = ["MAE", "RMSE", "R²"]

couleurs = [
    "#A8B5C2", "#71899E",
    "#087E8B", "#8B5EAD",
    "#39A5AC", "#B08CC5"]


fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)

for ax, metrique in zip(axes, ["MAE", "RMSE", "R²"]):
    barres = ax.barh(
        scores["Modèle"],
        scores[metrique],
        color=couleurs,
        height=0.6
    )

    ax.bar_label(
        barres,
        labels=[
            f"{valeur:.2f}".replace(".", ",")
            for valeur in scores[metrique]
        ],
        padding=5,
        fontsize=11
    )

    ax.set_title(f"Comparaison des {metrique}")
    ax.set_xlabel(f"{metrique} (unités du taux)")

    maximum = scores[metrique].max()
    ax.set_xlim(0, maximum * 1.20 if maximum > 0 else 1)

    ax.spines[["top", "right"]].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(axis="x", alpha=0.2)

# Premier modèle en haut ; l'axe est partagé entre les deux graphiques
axes[0].invert_yaxis()

fig.suptitle("Performances des modèles — Validation 2023")
plt.tight_layout()
plt.show()

In [ ]:

# Comparaison réel vs prédictions

# Choisir le département
dept = "13"

# Regrouper les prédictions
comparaison = valid[["code_dept", "semaine_a_prevoir"]].copy()

comparaison["Réel"] = y_valid
comparaison["Persistance"] = prediction_persistance
comparaison["ElasticNet A"] = prediction_A
comparaison["ElasticNet B"] = prediction_B
comparaison["ElasticNet A opt"] = prediction_A_opt
comparaison["ElasticNet B opt"] = prediction_B_opt

# Filtrer et trier les semaines
courbes = comparaison[
    comparaison["code_dept"] == dept
].sort_values("semaine_a_prevoir")

# Configuration des graphiques
groupes = [
    (["ElasticNet A", "ElasticNet A opt"],
     "Modèle A — Effet de l'optimisation"),

    (["ElasticNet B", "ElasticNet B opt"],
     "Modèle B — Effet de l'optimisation"),

    (["ElasticNet A opt", "ElasticNet B opt"],
     "Comparaison des modèles optimisés")
]

couleurs = {
    "ElasticNet A": "#087E8B",
    "ElasticNet A opt": "#39A5AC",
    "ElasticNet B": "#8B5EAD",
    "ElasticNet B opt": "#B08CC5"
}

fig, axes = plt.subplots(
    3, 1, figsize=(16, 12),
    sharex=True, sharey=True
)

for ax, (modeles, titre) in zip(axes, groupes):

    # Valeurs réelles
    ax.plot(
        courbes["semaine_a_prevoir"],
        courbes["Réel"],
        label="Réel",
        color="#E76F51",
        linewidth=2.5,
        zorder=3
    )

    # Baseline de persistance
    ax.plot(
        courbes["semaine_a_prevoir"],
        courbes["Persistance"],
        label="Persistance",
        color="#A8B5C2",
        linestyle="--",
        linewidth=1.5,
        alpha=0.7
    )

    # Modèles à comparer
    for modele in modeles:
        ax.plot(
            courbes["semaine_a_prevoir"],
            courbes[modele],
            label=modele,
            color=couleurs[modele],
            linewidth=2,
            linestyle="--" if "opt" in modele else "-"
        )

    ax.set_title(titre, fontsize=12)
    ax.set_ylabel("Taux pour 100 000 passages")
    ax.legend(loc="upper right", ncol=4, fontsize=9)
    ax.grid(alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)

# Afficher une semaine sur quatre
semaines = courbes["semaine_a_prevoir"].iloc[::4]

axes[-1].set_xticks(semaines)
axes[-1].tick_params(axis="x", rotation=45)
axes[-1].set_xlabel("Semaine cible (à prévoir)")

fig.suptitle(
    f"Prévisions vs valeurs réelles — Département {dept} — 2023",
    fontsize=15
)

plt.tight_layout()
plt.show()
